In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../")))
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from groq import Groq
from db.qdrant_setup import COLLECTION_NAME
from dotenv import load_dotenv
import qdrant_client as qc
from rank_bm25 import BM25Okapi
from collections import defaultdict
from sentence_transformers import CrossEncoder
import re
from collections import defaultdict
from fastembed import SparseTextEmbedding
import random

load_dotenv() 

qdrant = QdrantClient(path="./qdrant_storage")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
groq_client  = Groq(api_key=os.getenv("GROQ_API_KEY"))
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)


In [ ]:
class RAGState(TypedDict):
    question: str
    query_vector: List[float]
    retrieved_chunks: List[dict]
    relevance_scores: List[str]
    rewrite_count: int
    generation_attempts: int        # NEW: separate counter for hallucination retries
    prompt: str
    answer: str
    hallucination_check: str
    dense_chunks: List[dict]
    sparse_chunks: List[dict]
    query_type: str

In [ ]:
# ── Node 1: Embed the question ────────────────────────────────────
def query_classifier_node(state: RAGState):
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": """
You are a query classifier.

Classify the user's question into exactly ONE category.

Categories:

retrieval
- Questions that require searching the document.
- Questions about facts in uploaded documents.
- Questions asking "what", "who", "when", "where", "explain", "summarize", etc.

greeting
- Hi
- Hello
- Good morning
- Good evening
- Thanks
- Bye

chit_chat
- How are you?
- Tell me a joke.
- Who created you?
- What's your favorite color?

unsupported
- Harmful requests
- Illegal requests
- Anything outside assistant capability.

Return ONLY one word:

retrieval
greeting
chit_chat
unsupported
"""
            },
            {
                "role": "user",
                "content": state["question"]
            }
        ]
    )

    query_type = response.choices[0].message.content.strip().lower()

    print(f"📌 Query Type: {query_type}")

    return {
        "query_type": query_type
    }

def route_after_classifier(state: RAGState):
    query_type = state.get("query_type", "").lower()

    if query_type == "retrieval":
        return "embed"

    elif query_type == "greeting":
        return "general_llm"

    elif query_type == "chit_chat":
        return "general_llm"

    elif query_type == "unsupported":
        return "unsupported"

    return "general_llm"

def unsupported_node(state: RAGState):
    return {
        "answer": "Sorry, I can't assist with that request."
    }


def general_response_node(state: RAGState):
    query_type = state["query_type"]
    question = state["question"].lower().strip()

    greetings = [
        "Hello! How can I help you today?",
        "Hi! What can I do for you?",
        "Hey! How may I assist you?",
        "Welcome! Feel free to ask me anything."
    ]

    thanks = [
        "You're welcome!",
        "Happy to help!",
        "Glad I could help!",
        "Anytime!"
    ]

    if query_type == "greeting":
        if any(word in question for word in ["thank", "thanks"]):
            answer = random.choice(thanks)
        else:
            answer = random.choice(greetings)

    elif query_type == "chit_chat":
        answer = (
            "I'm designed to answer questions related to the uploaded documents. "
            "Please ask a document-related question."
        )

    else:
        answer = "I'm unable to process that request."

    return {"answer": answer}
def embed_node(state: RAGState):
    print(f"\n🔍 Embedding question: {state['question']}")

    vector = embed_model.encode(state["question"]).tolist()

    return {"query_vector": vector}


def dense_retrieve_node(state: RAGState):
    print(f"\n🔍 Dense retrieval for: {state['question']}")

    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=state["query_vector"],
        limit=10,
        with_payload=True,
    ).points

    dense_chunks = []
    for hit in results:
        dense_chunks.append({
            "id": str(hit.id),
            "text": hit.payload.get("text"),
            "heading": hit.payload.get("heading"),
            "page_start": hit.payload.get("page_start"),
            "dense_score": float(hit.score),
        })

    return {"dense_chunks": dense_chunks}

def sparse_retrieve_node(state: RAGState):
    print(f"🔍 Sparse retrieval for: {state['question']}")

    query_text = " ".join(preprocess(state["question"]))
    query_sparse = list(sparse_model.embed(query_text))[0]

    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_sparse,
        using="bm25",
        limit=5,
        with_payload=True,
    ).points

    sparse_chunks = []
    for hit in results:
        sparse_chunks.append({
            "id": str(hit.id),
            "text": hit.payload["text"],
            "heading": hit.payload.get("heading"),
            "page_start": hit.payload.get("page_start"),
            "bm25_score": float(hit.score),
        })

    return {"sparse_chunks": sparse_chunks}
    
def fusion_node(state: RAGState):
    fused_scores = defaultdict(float)
    chunk_lookup = {}

    for rank, chunk in enumerate(state["dense_chunks"]):
        fused_scores[chunk["id"]] += 1 / (rank + 1)
        chunk_lookup[chunk["id"]] = chunk

    for rank, chunk in enumerate(state["sparse_chunks"]):
        fused_scores[chunk["id"]] += 1 / (rank + 1)
        chunk_lookup[chunk["id"]] = chunk

    ranked_ids = sorted(fused_scores, key=fused_scores.get, reverse=True)

    final_chunks = []
    for chunk_id in ranked_ids[:10]:
        chunk = chunk_lookup[chunk_id]
        chunk["hybrid_score"] = round(fused_scores[chunk_id], 4)
        final_chunks.append(chunk)

    print("\n📄 Hybrid Retrieval Results\n")
    for i, chunk in enumerate(final_chunks, 1):
        print(f"{i}. [{chunk['hybrid_score']}] {chunk.get('heading') or 'No heading'}")

    return {"retrieved_chunks": final_chunks}

def grade_docs_node(state: RAGState):
    scores = []
    relevant_chunks = []

    for chunk in state["retrieved_chunks"]:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{
                "role": "user",
                "content": f"""
                    You are a retrieval evaluator.
                    A document is relevant if it contains information
                    that helps answer the question directly or partially.
                    Question: {state['question']}
                    Document: {chunk['text'][:400]}
                    Answer ONLY 'relevant' or 'irrelevant'."""
            }],
            temperature=0,
        )
        verdict = response.choices[0].message.content.strip().lower()
        scores.append(verdict)
        if "relevant" in verdict and "irrelevant" not in verdict:
            relevant_chunks.append(chunk)

    print(f"📊 Relevance scores: {scores}")
    return {
        "relevance_scores": scores,
        "retrieved_chunks": relevant_chunks,   # only relevant chunks flow forward
    }
def dense_retrieve_node(state: RAGState):

    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=state["query_vector"],
        limit=5,
        with_payload=True,
    ).points

    dense_chunks = []

    for hit in results:
        dense_chunks.append({
            "id": str(hit.id),
            "text": hit.payload.get("text"),
            "heading": hit.payload.get("heading"),
            "page_start": hit.payload.get("page_start"),
            "dense_score": float(hit.score),
        })

    return {"dense_chunks": dense_chunks}


def fusion_node(state: RAGState):
    """
    Weighted Reciprocal Rank Fusion (WRRF)

    Dense retrieval has higher importance than sparse retrieval.
    """

    # Adjust these weights based on your experiments
    DENSE_WEIGHT = 0.7
    SPARSE_WEIGHT = 0.3

    # Standard RRF constant
    K = 60

    fused_scores = defaultdict(float)
    chunk_lookup = {}

    # ---------------------------
    # Dense Retrieval Contribution
    # ---------------------------
    for rank, chunk in enumerate(state["dense_chunks"]):

        score = DENSE_WEIGHT * (1 / (rank + K))

        fused_scores[chunk["id"]] += score

        chunk_lookup[chunk["id"]] = chunk

    # ----------------------------
    # Sparse Retrieval Contribution
    # ----------------------------
    for rank, chunk in enumerate(state["sparse_chunks"]):

        score = SPARSE_WEIGHT * (1 / (rank + K))

        fused_scores[chunk["id"]] += score

        if chunk["id"] not in chunk_lookup:
            chunk_lookup[chunk["id"]] = chunk

    # ----------------------------
    # Sort by fused score
    # ----------------------------
    ranked_ids = sorted(
        fused_scores.keys(),
        key=lambda x: fused_scores[x],
        reverse=True,
    )

    final_chunks = []

    for chunk_id in ranked_ids[:10]:

        chunk = chunk_lookup[chunk_id].copy()

        chunk["hybrid_score"] = round(
            fused_scores[chunk_id],
            6
        )

        final_chunks.append(chunk)

    print("\n========== Hybrid Retrieval ==========\n")

    for i, chunk in enumerate(final_chunks, 1):

        print(
            f"{i}. "
            f"Score={chunk['hybrid_score']:.6f} | "
            f"Dense={chunk.get('dense_score', 0):.4f} | "
            f"Sparse={chunk.get('bm25_score', 0):.4f} | "
            f"{chunk.get('heading', 'No Heading')}"
        )

    print("\n======================================\n")

    return {
        "retrieved_chunks": final_chunks
    }



def rerank_node(state: RAGState, top_k: int = 10):

    chunks = state.get("retrieved_chunks", [])

    if not chunks:
        return {"retrieved_chunks": []}

    pairs = [
        (state["question"], chunk["text"])
        for chunk in chunks
    ]

    scores = reranker.predict(
        pairs,
        batch_size=32,
        show_progress_bar=False
    )

    for chunk, score in zip(chunks, scores):
        chunk["rerank_score"] = float(score)

    reranked = sorted(
        chunks,
        key=lambda x: x["rerank_score"],
        reverse=True
    )[:top_k]

    return {
        "retrieved_chunks": reranked
    }

def rewrite_node(state: RAGState):
    if state["rewrite_count"] >= 2:          # hard limit on retries
        print("⚠️ Max rewrites reached, using original")
        return {"rewrite_count": state["rewrite_count"]}

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""Rewrite this question to improve document retrieval.
Original: {state['question']}
Return ONLY the rewritten question."""
        }],
        temperature=0.3,
    )
    new_q = response.choices[0].message.content.strip()
    new_vec = embed_model.encode(new_q).tolist()
    print(f"🔄 Rewritten: {new_q}")
    return {
        "question": new_q,
        "query_vector": new_vec,
        "rewrite_count": state["rewrite_count"] + 1
    }




def prompt_node(state: RAGState):
    context_parts = []

    for i, chunk in enumerate(state["retrieved_chunks"], 1):
        heading = f"[{chunk['heading']}]" if chunk["heading"] else ""
        page = f"(Page {chunk['page_start']})" if chunk["page_start"] else ""
        context_parts.append(
            f"Chunk {i} {heading} {page}:\n{chunk['text']}"
        )

    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know based on the document."

CONTEXT:
{context}

QUESTION:
{state['question']}

ANSWER:"""

    return {"prompt": prompt}




# ── Node 4: LLM response via Groq ────────────────────────────────
def llm_node(state: RAGState):
    print("\n🤖 Answer:\n")

    completion = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": "You are a document assistant. Answer only from the provided context."},
            {"role": "user", "content": state["prompt"]}
        ],
        temperature=0.2,
        top_p=1,
        reasoning_effort="medium",
        stream=True,
        stop=None
    )

    full_answer = ""
    for chunk in completion:
        token = chunk.choices[0].delta.content or ""
        print(token, end="", flush=True)
        full_answer += token

    print("\n")
    return {
        "answer": full_answer,
        "generation_attempts": state.get("generation_attempts", 0) + 1,   # NEW
    }



def hallucination_check_node(state: RAGState):
    context = "\n".join(c["text"][:500] for c in state["retrieved_chunks"])
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""Is this answer grounded in the context?
Context: {context}
Answer: {state['answer']}
Reply ONLY 'grounded' or 'hallucinated'."""
        }],
        temperature=0,
    )
    result = response.choices[0].message.content.strip().lower()
    print(f"🧠 Hallucination check: {result}")
    return {"hallucination_check": result}

def no_answer_node(state: RAGState):
    print("❌ Question is not relevant to this document.")
    return {"answer": "I don't know based on the document."}

def route_after_grading(state: RAGState):
    if "relevant" in state["relevance_scores"]:
        return "prompt"                    
    if state["rewrite_count"] >= 2:
        return "no_answer"                  

    return "rewrite"            

def route_after_hallucination(state: RAGState):
    if state["hallucination_check"] == "grounded":
        return END
    if state.get("generation_attempts", 0) >= 2:
        return END
    return "prompt"



In [ ]:

# ── Build Graph ───────────────────────────────────────────────────

graph = StateGraph(RAGState)

# Retrieval Pipeline
graph.add_node("query_classifier_node", query_classifier_node)
graph.add_node("unsupported_node", unsupported_node)
graph.add_node("general_response_node", general_response_node)

graph.add_node("route_after_classifier", route_after_classifier)

graph.add_node("embed", embed_node)
graph.add_node("dense_retrieve", dense_retrieve_node)
graph.add_node("sparse_retrieve", sparse_retrieve_node)
graph.add_node("fusion", fusion_node)
graph.add_node("rerank", rerank_node)

# RAG Nodes
graph.add_node("grade", grade_docs_node)
graph.add_node("rewrite", rewrite_node)
graph.add_node("prompt", prompt_node)
graph.add_node("llm", llm_node)
graph.add_node("hallcheck", hallucination_check_node)
graph.add_node("no_answer", no_answer_node)

# ── Main Flow ───────────────────────────────────────────────────

graph.add_edge(START, "query_classifier_node")
graph.add_conditional_edges(
    "query_classifier_node",
    route_after_classifier,
    {
        "embed": "embed",
        "general_response": "general_response",
    }
)

graph.add_edge("embed", "dense_retrieve")
graph.add_edge("dense_retrieve", "sparse_retrieve")
graph.add_edge("sparse_retrieve", "fusion")
graph.add_edge("fusion", "rerank")
graph.add_edge("rerank", "grade")

# ── Relevance Routing ───────────────────────────────────────────

graph.add_conditional_edges(
    "grade",
    route_after_grading,
    {
        "prompt": "prompt",
        "rewrite": "rewrite",
        "no_answer": "no_answer",
    }
)

# Rewrite Loop
graph.add_edge("rewrite", "embed")

# ── Generation ──────────────────────────────────────────────────

graph.add_edge("prompt", "llm")
graph.add_edge("llm", "hallcheck")

# ── Hallucination Routing ───────────────────────────────────────

graph.add_conditional_edges(
    "hallcheck",
    route_after_hallucination,
    {
        "prompt": "prompt",
        END: END,
    }
)

graph.add_edge("no_answer", END)

# ── Compile ─────────────────────────────────────────────────────

rag_app = graph.compile()

# ── Run ─────────────────────────────────────────────────────────

result = rag_app.invoke({
    "question": "who is gandhi?",
    "query_vector": [],
    "dense_chunks": [],
    "sparse_chunks": [],
    "retrieved_chunks": [],
    "relevance_scores": [],
    "hallucination_check": "",
    "prompt": "",
    "answer": "",
    "rewrite_count": 0,
    "generation_attempts": 0,   # NEW
})

print("\n✅ Final Answer Stored:\n")
print(result["answer"])



In [ ]:

# Visualize Graph

from IPython.display import Image, display

display(
    Image(
        rag_app.get_graph().draw_mermaid_png()
    )
)